# Validation ROI Analysis

This notebook investigates why average ROI@10 is negative across tuning models for validation splits 1 and 2.
The hypothesis is that the negative ROI is driven by adverse market conditions (COVID-19 crash) during those windows, not by poor model quality.

In [ ]:
import math
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

ROOT = Path("..")
CLOSE_PRICES_PATH = ROOT / "data" / "original" / "close_prices.csv"
CALENDAR_DAYS_PER_MONTH = 30

## 1. Load Data

In [ ]:
def load_close_prices(path: Path) -> pd.DataFrame:
    """Load and parse close prices CSV."""
    close_prices = pd.read_csv(path, parse_dates=["timestamp"])
    close_prices["date"] = close_prices["timestamp"].dt.date
    return close_prices


close_prices = load_close_prices(CLOSE_PRICES_PATH)
print(f"Loaded {len(close_prices):,} price records")
print(f"Date range: {close_prices['date'].min()} to {close_prices['date'].max()}")
print(f"Unique assets: {close_prices['ISIN'].nunique():,}")

## 2. Define Validation Windows

The 3 validation splits used during hyperparameter tuning, each with a 6-month test window.

In [ ]:
VALIDATION_WINDOWS: list[tuple[str, date, date]] = [
    ("Split 0 (Apr 2019)",  date(2019, 4, 1),  date(2019, 10, 1)),
    ("Split 1 (Oct 2019)",  date(2019, 10, 1), date(2020, 4, 1)),
    ("Split 2 (Jan 2020)",  date(2020, 1, 31), date(2020, 7, 31)),
]

WINDOW_COLORS = ["#2ecc71", "#e74c3c", "#e67e22"]

## 3. Market Return Functions

In [ ]:
def get_price_on_or_before(asset_prices: pd.DataFrame, target: date) -> float | None:
    """Return the most recent close price on or before target date."""
    candidates = asset_prices[asset_prices["date"] <= target]
    if candidates.empty:
        return None
    return float(candidates.sort_values("date").iloc[-1]["closePrice"])


def compute_monthly_return(start_price: float, end_price: float, days: int) -> float:
    """Compute geometric monthly return matching the FAR-Trans ROI@k formula."""
    total_return = (end_price - start_price) / start_price
    return math.pow(1.0 + total_return, CALENDAR_DAYS_PER_MONTH / days) - 1.0


def compute_per_asset_returns(
    close_prices: pd.DataFrame,
    time_point: date,
    test_end: date,
) -> pd.DataFrame:
    """Compute per-asset monthly returns for a given time window.

    Returns a DataFrame with columns [ISIN, start_price, end_price,
    total_return, monthly_return] for assets with valid prices at both ends.
    """
    days = (test_end - time_point).days
    records: list[dict] = []

    for isin, group in close_prices.groupby("ISIN"):
        start = get_price_on_or_before(group, time_point)
        end = get_price_on_or_before(group, test_end)

        if start is None or end is None or start <= 0:
            continue

        total_return = (end - start) / start
        monthly_return = compute_monthly_return(start, end, days)

        records.append({
            "ISIN": isin,
            "start_price": start,
            "end_price": end,
            "total_return": total_return,
            "monthly_return": monthly_return,
        })

    return pd.DataFrame(records)


def compute_market_summary(
    close_prices: pd.DataFrame,
    windows: list[tuple[str, date, date]],
) -> pd.DataFrame:
    """Compute market-wide return statistics for each validation window."""
    rows: list[dict] = []

    for label, time_point, test_end in windows:
        asset_returns = compute_per_asset_returns(close_prices, time_point, test_end)

        rows.append({
            "window": label,
            "time_point": time_point,
            "test_end": test_end,
            "eligible_assets": len(asset_returns),
            "mean_total_return": asset_returns["total_return"].mean(),
            "median_total_return": asset_returns["total_return"].median(),
            "mean_monthly_return": asset_returns["monthly_return"].mean(),
            "median_monthly_return": asset_returns["monthly_return"].median(),
            "pct_assets_negative": (asset_returns["total_return"] < 0).mean() * 100,
        })

    return pd.DataFrame(rows)

In [ ]:
summary = compute_market_summary(close_prices, VALIDATION_WINDOWS)

display_cols = [
    "window", "time_point", "test_end", "eligible_assets",
    "mean_total_return", "median_total_return",
    "mean_monthly_return", "median_monthly_return",
    "pct_assets_negative",
]
summary[display_cols].style.format({
    "mean_total_return": "{:.2%}",
    "median_total_return": "{:.2%}",
    "mean_monthly_return": "{:.4f}",
    "median_monthly_return": "{:.4f}",
    "pct_assets_negative": "{:.1f}%",
})

## 4. Market Index Over Time

Equal-weighted index of all assets with prices on 2019-01-02 (first available date before the validation windows), normalized to 1.0.

In [ ]:
def build_equal_weight_market_index(
    close_prices: pd.DataFrame,
    start_date: date,
    end_date: date,
) -> pd.Series:
    """Build an equal-weighted market price index normalized to 1.0 at start_date.

    Only includes assets with a valid price on or before start_date.
    For each trading date, computes the mean close price across the universe,
    then normalizes by the universe mean at start_date.
    """
    universe = {
        isin
        for isin, group in close_prices.groupby("ISIN")
        if get_price_on_or_before(group, start_date) is not None
    }

    window_prices = close_prices[
        (close_prices["ISIN"].isin(universe))
        & (close_prices["date"] >= start_date)
        & (close_prices["date"] <= end_date)
    ].copy()

    daily_mean = window_prices.groupby("date")["closePrice"].mean().sort_index()
    return daily_mean / daily_mean.iloc[0]


INDEX_START = date(2018, 1, 2)
INDEX_END = date(2021, 6, 30)

market_index = build_equal_weight_market_index(close_prices, INDEX_START, INDEX_END)
print(f"Market index: {len(market_index)} trading days from {INDEX_START} to {INDEX_END}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [3, 2]})

ax_index = axes[0]
ax_dist = axes[1]

ax_index.plot(
    [pd.Timestamp(d) for d in market_index.index],
    market_index.values,
    color="#2c3e50",
    linewidth=1.5,
    label="Equal-weight market index",
)
ax_index.axhline(1.0, color="grey", linestyle="--", linewidth=0.8, alpha=0.6)

legend_patches = []
for (label, time_point, test_end), color in zip(VALIDATION_WINDOWS, WINDOW_COLORS):
    ax_index.axvspan(
        pd.Timestamp(time_point), pd.Timestamp(test_end),
        alpha=0.2, color=color,
    )
    ax_index.axvline(pd.Timestamp(time_point), color=color, linewidth=1.2, linestyle=":")
    ax_index.axvline(pd.Timestamp(test_end), color=color, linewidth=1.2, linestyle=":")
    legend_patches.append(mpatches.Patch(color=color, alpha=0.5, label=label))

ax_index.annotate(
    "COVID-19\nmarket crash",
    xy=(pd.Timestamp("2020-03-23"), market_index.asof(date(2020, 3, 23))),
    xytext=(pd.Timestamp("2020-06-01"), 0.55),
    arrowprops=dict(arrowstyle="->", color="black"),
    fontsize=9,
)

ax_index.set_title("Equal-Weight Market Index with Validation Windows", fontsize=13)
ax_index.set_ylabel("Normalized Price (base = 1.0 at 2018-01-02)")
ax_index.legend(handles=[ax_index.lines[0]] + legend_patches, loc="upper left", fontsize=9)
ax_index.set_xlim(pd.Timestamp(INDEX_START), pd.Timestamp(INDEX_END))
ax_index.grid(axis="y", alpha=0.3)

all_returns: list[pd.Series] = []
window_labels: list[str] = []

for (label, time_point, test_end), color in zip(VALIDATION_WINDOWS, WINDOW_COLORS):
    asset_returns = compute_per_asset_returns(close_prices, time_point, test_end)
    clipped = asset_returns["total_return"].clip(-1.0, 2.0)
    all_returns.append(clipped)
    window_labels.append(label)

bp = ax_dist.boxplot(
    [r.values for r in all_returns],
    labels=window_labels,
    patch_artist=True,
    medianprops=dict(color="black", linewidth=2),
    showfliers=False,
)
for patch, color in zip(bp["boxes"], WINDOW_COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax_dist.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax_dist.set_title("Distribution of Per-Asset 6-Month Total Returns per Validation Window", fontsize=11)
ax_dist.set_ylabel("Total Return (clipped to [-100%, +200%])")
ax_dist.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax_dist.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / "outputs" / "roi_validation_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Return Statistics vs Tuning ROI Results

Compare the market-wide average monthly return against the best-trial average ROI reported by Random Forest, LightGCN, and SASRec.
Each model's value is the `average_roi` from its best hyperparameter configuration (selected by primary metric) averaged across all three validation splits.

In [ ]:
TUNING_ROI_RESULTS: dict[str, float] = {
    "Random Forest": -3.533451703314518e-05,
    "LightGCN":      -0.04373439211726443,
    "SASRec":        -0.048978609835443644,
}


def compute_market_monthly_return(
    close_prices: pd.DataFrame,
    time_point: date,
    test_end: date,
) -> float:
    """Compute market-wide mean monthly return for the window, matching ROI@k formula."""
    asset_returns = compute_per_asset_returns(close_prices, time_point, test_end)
    return float(asset_returns["monthly_return"].mean())


market_monthly_returns: list[float] = [
    compute_market_monthly_return(close_prices, time_point, test_end)
    for _, time_point, test_end in VALIDATION_WINDOWS
]
average_market_roi: float = sum(market_monthly_returns) / len(market_monthly_returns)

comparison_data: dict[str, float] = {"Market Average": average_market_roi}
comparison_data.update(TUNING_ROI_RESULTS)

comparison_df = pd.DataFrame([
    {"model": name, "average_monthly_roi": roi}
    for name, roi in comparison_data.items()
]).set_index("model")

comparison_df.style.format("{:.4f}").applymap(
    lambda v: "color: red" if v < 0 else "color: green"
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

display_names = list(comparison_data.keys())
roi_values = list(comparison_data.values())
bar_colors = ["#95a5a6", "#3498db", "#9b59b6", "#e74c3c"]

ax.barh(
    display_names,
    roi_values,
    color=bar_colors[: len(display_names)],
    alpha=0.8,
)

ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Average Monthly ROI (across all validation windows)")
ax.set_title("Average Monthly ROI: Market vs Best-Trial Tuned Models")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.4f}"))
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Summary

Key findings from this analysis:

In [ ]:
print("Market conditions per validation window:")
for label, time_point, test_end in VALIDATION_WINDOWS:
    asset_returns = compute_per_asset_returns(close_prices, time_point, test_end)
    pct_neg = (asset_returns["total_return"] < 0).mean() * 100
    mean_ret = asset_returns["total_return"].mean()
    print(f"  {label} ({time_point} to {test_end}):")
    print(f"    Mean market total return: {mean_ret:+.2%}")
    print(f"    Assets with negative return: {pct_neg:.1f}%")

print(f"\nMarket average monthly ROI (across all windows): {average_market_roi:+.4f}")
print("\nBest-trial average model ROI (across all windows):")
for model_name, avg_roi in TUNING_ROI_RESULTS.items():
    print(f"  {model_name}: {avg_roi:+.4f}")